# Example 1 - Download and read CVR data

This notebook explains step-by-step how to download the data locally with minimal terminal interaction. 

The few commands runned can be replaced using any file explorer in your computer.

You will need to:
1. API credentials given by virk.dk. In practice this takes 3 min to send an email and wait a couple of days. Instructions in the `data_extraction/README.md` file.
2. Have installed `git`, 3.10+ `python`, a python virtual environment (e.g. `conda` or `uv`).

In [18]:
!git --version

git version 2.50.1 (Apple Git-155)


In [19]:
!python --version

Python 3.12.13


In [5]:
import os 
import json
import pandas as pd

## 1. Get the scripts

In [11]:
WORKING_DIR_PATH="/Users/pipegalera/dev/GitHub/"

os.chdir(MY_DESKTOP_PATH)
os.getcwd()

'/Users/pipegalera/dev/GitHub/CVR_data'

In [2]:
#!git clone https://github.com/CBS-SI/CVR_data.git

Cloning into 'CVR_data'...
remote: Enumerating objects: 289, done.
remote: Counting objects: 100% (289/289), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 289 (delta 144), reused 236 (delta 91), pack-reused 0 (from 0)
Receiving objects: 100% (289/289), 2.99 MiB | 12.35 MiB/s, done.
Resolving deltas: 100% (144/144), done.


In [12]:
!ls | grep CVR_data

In [13]:
os.chdir(f"{WORKING_DIR_PATH}/CVR_data")
os.getcwd()

'/Users/pipegalera/dev/GitHub/CVR_data'

In [14]:
!ls

LICENSE                 examples                notebook_tests
README.md               flask_app               raw_data
data_extraction         images                  testing_environment.yml
environment.yml         kestra                  utils


Please install the `environment.yml` (e.g. `conda create -f environment.yml`).  

`testing_environment.yml` includes jupyter/ipykernel.

## 2. Set up the environment

Please notice that the script do not work without: 
1. The virk.dk API creds.
2. Setting the storage folders path.

Here I will create an `.env` file and copy the creds manually. `raw_data`/`proc_data` can be whatever folder you like.

In [26]:
!touch .env

In [27]:
!mkdir raw_data 
!mkdir proc_data

Copy in the .env file the creds and folder paths:

```py
# Secrets
VIRK_USERNAME = "username provided by CVR office"
VIRK_PASSWORD = "password provided by CVR office"

# Storage
RAW_VIRKSOMHED_FOLDER_PATH = "/Users/pg/Desktop/CVR_data/raw_data" 
PROC_VIRKSOMHED_FOLDER_PATH = "/Users/pg/Desktop/CVR_data/proc_data"
```

## 3. Download data

The script takes useful arguments: 

```python
# Backfill everything (founding years 1800 .. current year, plus the "unknown" partition)
python get_historical_virksomhed_api_call.py

# Backfill a specific founding-year range (inclusive on both ends)
python get_historical_virksomhed_api_call.py --start-year 1990 --end-year 2025

# Re-download years that are already present on disk
python get_historical_virksomhed_api_call.py --overwrite

# Fetch ONLY companies with no founding date (the "unknown" partition)
python get_historical_virksomhed_api_call.py --year-unknown

# Write to a specific output folder (defaults to RAW_VIRKSOMHED_FOLDER_PATH in .env)
python get_historical_virksomhed_api_call.py --folder /path/to/output
```

**Read the documentation under data_extraction for more details**

In [43]:
!ls

LICENSE                 examples                testing_environment.yml
README.md               images                  utils
data_extraction         proc_data
environment.yml         raw_data


I will choose to download the companies founded in 1980 until 1981 (included). I purposely selected this year because they are not that many companies. 

In [29]:
!python data_extraction/src/get_historical_virksomhed_api_call.py --start-year 1980 --end-year 1981

Fetching founding year: 1980...
Total hits to fetch: 8851
Fetched 6000/8851
Fetched 8851/8851
-> Saving 8851 records in parquet files for 1980...
Fetching founding year: 1981...
Total hits to fetch: 9990
Fetched 6000/9990
Fetched 9000/9990
Fetched 9990/9990
-> Saving 9990 records in parquet files for 1981...
-> Download complete. Parquet files saved in /Users/pg/Desktop/CVR_data/raw_data.
-> Timestamp set to 2026-06-07 11:26:37 UTC and stored as `_state.json` within the same folder.
   14.02 s in run(['1980', '1981'], '/Users/pg/Desktop/CVR..., False)


The python script downloads 20 files (CVR tables) per year. They are named as `<table>_<year>.parquet`

In [45]:
!ls raw_data

_state.json                         livsforloeb_1980.parquet
aarsbeskaeftigelse_1980.parquet     livsforloeb_1981.parquet
aarsbeskaeftigelse_1981.parquet     maanedsbeskaeftigelse_1980.parquet
beliggenhedsadresse_1980.parquet    maanedsbeskaeftigelse_1981.parquet
beliggenhedsadresse_1981.parquet    main_1980.parquet
bibranche1_1980.parquet             main_1981.parquet
bibranche1_1981.parquet             navne_1980.parquet
bibranche2_1980.parquet             navne_1981.parquet
bibranche2_1981.parquet             postadresse_1980.parquet
bibranche3_1980.parquet             postadresse_1981.parquet
bibranche3_1981.parquet             regNummer_1980.parquet
binavne_1980.parquet                regNummer_1981.parquet
binavne_1981.parquet                telefaxNummer_1980.parquet
elektroniskPost_1980.parquet        telefaxNummer_1981.parquet
elektroniskPost_1981.parquet        telefonNummer_1980.parquet
hjemmeside_1980.parquet             telefonNummer_1981.parquet
hjemmeside_1981.parquet   

Notice that t includes a `_state.json` file in case you want to update the data in the future:

In [15]:
with open('raw_data/_state.json', 'r') as json_file:
    parsed_json = json.load(json_file)
    print(json.dumps(parsed_json, indent=2))

{
  "years": {
    "1980": "2026-06-07T15:45:44.063662+00:00",
    "1981": "2026-06-07T15:45:44.063662+00:00",
    "1978": "2026-06-07T21:13:15.972096+00:00",
    "1979": "2026-06-07T21:13:15.972096+00:00",
    "1992": "2026-06-07T21:13:15.972096+00:00",
    "1993": "2026-06-07T21:13:15.972096+00:00",
    "1994": "2026-06-07T21:13:15.972096+00:00",
    "1995": "2026-06-07T21:13:15.972096+00:00",
    "1996": "2026-06-07T21:14:48.558272+00:00",
    "1997": "2026-06-07T21:14:48.558272+00:00",
    "1998": "2026-06-07T21:14:48.558272+00:00",
    "1999": "2026-06-07T21:14:48.558272+00:00",
    "2000": "2026-06-07T21:14:48.558272+00:00",
    "2001": "2026-06-07T21:18:25.204281+00:00",
    "2002": "2026-06-07T21:18:25.204281+00:00",
    "2003": "2026-06-07T21:18:25.204281+00:00",
    "2004": "2026-06-07T21:18:25.204281+00:00",
    "2005": "2026-06-07T21:18:25.204281+00:00",
    "2006": "2026-06-07T21:39:36.048261+00:00",
    "2007": "2026-06-07T21:39:36.048261+00:00",
    "2008": "2026-06-07T2

## 4. Update data 

Update script run using the key file `_state.json` you saw before. It uses to now when was the last time you run the script, adds a bit of a buffer (1 day by default), and run it from there. 

Also takes useful arguments:

```python
    # by default every founding year on disk is refreshed.
    python update_data.py

    # --start-year and --end-year to refresh only those year set.
    python update_data.py --start-year 1980 --end-year 2005

    # --year-unknown to refresh only the companies with no founding year date.
    python update_data.py --year-unknown

    # --since to override the start date for the selected years, use with care
    python update_data.py --since 2026-06-01

    # --buffer-days to change the default time lookback buffer.
    python update_data.py --buffer-days 2
```

In [53]:
!python data_extraction/src/update_data_virksomhed_api_call.py --start-year 1980 --end-year 1981

Updating 2 founding year(s) changed since 2026-06-06 (buffer 1d)
Total hits to fetch: 1
1 changed companies
Upserting year 1980: 1 companies
Update complete. 2 year(s) now current as of 2026-06-07T11:41:04.815796+00:00


In [54]:
with open('raw_data/_state.json', 'r') as json_file:
    parsed_json = json.load(json_file)
    print(json.dumps(parsed_json, indent=2))

{
  "years": {
    "1980": "2026-06-07T11:41:04.815796+00:00",
    "1981": "2026-06-07T11:41:04.815796+00:00"
  },
  "last_run_utc": "2026-06-07T11:41:04.815796+00:00"
}


Notice that the dates are now updated to the last run. 

It updated 1 company info (founded in 1980). This is not because the company info got updated in the last minite, the script picked up a change that was registered yesterday in virk (--buffer-days=1).

Normally old companies are very big (e.g. `Novo Nordisk`, `Maersk`, `DFS`...) and get updated constantly in `virk.dk`. The scripts picks up the changes 

## 5. Read the data

The parque files are indexed by `cvr` number. 

Please follow the rules using this data (read [this section of the readme](https://github.com/CBS-SI/CVR_data#gdpr-and-data-protection)).

In [16]:
!ls raw_data

_state.json                         livsforloeb_1800.parquet
aarsbeskaeftigelse_1800.parquet     livsforloeb_1801.parquet
aarsbeskaeftigelse_1801.parquet     livsforloeb_1802.parquet
aarsbeskaeftigelse_1802.parquet     livsforloeb_1803.parquet
aarsbeskaeftigelse_1803.parquet     livsforloeb_1804.parquet
aarsbeskaeftigelse_1804.parquet     livsforloeb_1805.parquet
aarsbeskaeftigelse_1805.parquet     livsforloeb_1806.parquet
aarsbeskaeftigelse_1806.parquet     livsforloeb_1807.parquet
aarsbeskaeftigelse_1807.parquet     livsforloeb_1808.parquet
aarsbeskaeftigelse_1808.parquet     livsforloeb_1809.parquet
aarsbeskaeftigelse_1809.parquet     livsforloeb_1810.parquet
aarsbeskaeftigelse_1810.parquet     livsforloeb_1811.parquet
aarsbeskaeftigelse_1811.parquet     livsforloeb_1812.parquet
aarsbeskaeftigelse_1812.parquet     livsforloeb_1813.parquet
aarsbeskaeftigelse_1813.parquet     livsforloeb_1814.parquet
aarsbeskaeftigelse_1814.parquet     livsforloeb_1815.parquet
aarsbeskaeftigelse_1815.

In [17]:
file = "kvartalsbeskaeftigelse_2012.parquet"

pd.read_parquet(f"raw_data/{file}")

,cvrNummer,enhedsNummer,aar,kvartal,antalAarsvaerk,antalAnsatte,sidstOpdateret,intervalKodeAntalAarsvaerk,intervalKodeAntalAnsatte
0,34251118,4001922074,2012,3,1.0,NaN,2013-04-29T14:46:34.000+02:00,ANTAL_1_1,NaN
1,34251118,4001922074,2012,4,1.0,NaN,2013-07-18T14:52:02.000+02:00,ANTAL_1_1,NaN
2,34251118,4001922074,2013,1,1.0,NaN,2013-10-09T14:47:15.000+02:00,ANTAL_1_1,NaN
3,34251118,4001922074,2013,2,1.0,NaN,2014-01-23T22:37:47.000+01:00,ANTAL_1_1,NaN
4,34251118,4001922074,2013,3,1.0,NaN,2014-04-04T21:20:19.000+02:00,ANTAL_1_1,NaN
...,...,...,...,...,...,...,...,...,...
191093,35039244,4001943206,2015,3,3.0,9.0,2016-03-18T21:06:31.000Z,ANTAL_2_4,ANTAL_5_9
191094,35039244,4001943206,2015,4,3.0,10.0,2016-06-25T21:12:23.000Z,ANTAL_2_4,ANTAL_10_19
191095,35039244,4001943206,2016,1,1.0,0.0,2016-06-25T21:12:23.000Z,ANTAL_1_1,ANTAL_0_0
191096,35039244,4001943206,2019,2,3.0,13.0,2020-01-02T21:52:28.000Z,ANTAL_2_4,ANTAL_10_19
